# Stochastic Volatility Model Calibration

This notebook demonstrates calibration of **Heston** and **SABR** models on market data.

## Models
- **Heston (1993)**: `dS = sqrt(V)·S·dW₁`, `dV = κ(θ-V)dt + ξ√V·dW₂`, `corr = ρ`
- **SABR (Hagan 2002)**: `dF = α·Fᵝ·dW₁`, `dα = ν·α·dW₂`, `corr = ρ`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from models.heston import HestonModel, HestonParams
from models.sabr import SABRModel, SABRParams
from data.market_data import MarketDataFetcher
from calibration.heston_calibrator import HestonCalibrator
from calibration.sabr_calibrator import SABRCalibrator
from visualization.plotter import VolSurfacePlotter

## 1. Generate / Fetch Market Data

In [ ]:
# Option A: Synthetic data (always works, good for testing)
true_params = HestonParams(kappa=2.0, theta=0.04, xi=0.35, rho=-0.65, v0=0.04)
chain = MarketDataFetcher.synthetic_chain(spot=100.0, risk_free_rate=0.05, true_heston_params=true_params)

# Option B: Real data from Yahoo Finance (requires internet + yfinance)
# fetcher = MarketDataFetcher(risk_free_rate=0.05, dividend_yield=0.013)
# chain = fetcher.fetch('SPY', max_expirations=6).filter()

df = chain.to_dataframe()
print(f'Spot: {chain.spot:.2f} | Quotes: {len(chain.quotes)}')
df.groupby('T').size().rename('count').to_frame().T

## 2. Heston Calibration

In [ ]:
heston_calibrator = HestonCalibrator(weight_by_vega=True)
heston_result = heston_calibrator.calibrate(chain, popsize=12, maxiter=200, verbose=True)

## 3. SABR Calibration

In [ ]:
sabr_calibrator = SABRCalibrator(fix_beta=0.5, n_restarts=8)
sabr_result = sabr_calibrator.calibrate(chain, verbose=True)

## 4. Visualize Vol Smiles

In [ ]:
plotter = VolSurfacePlotter(dark_mode=False)
fig = plotter.plot_smile_fit(chain, heston_result)
plt.show()

## 5. 3D Implied Volatility Surface

In [ ]:
fig = plotter.plot_surface_3d(chain, heston_result)
plt.show()

## 6. Model Comparison

In [ ]:
results = {'Heston': heston_result, 'SABR': sabr_result}
fig = plotter.plot_comparison(chain, results, T_target=0.25)
plt.show()

print(f"\nHeston RMSE: {heston_result.rmse*100:.4f}%")
print(f"SABR   RMSE: {sabr_result.rmse*100:.4f}%")

## 7. Heston Parameter Sensitivity

How does each parameter affect the vol smile shape?

In [ ]:
base = HestonParams(kappa=2.0, theta=0.04, xi=0.3, rho=-0.7, v0=0.04)
S, T, r = 100.0, 0.5, 0.05
strikes = np.linspace(80, 120, 50)

param_ranges = {
    'rho':   ('rho',   [-0.9, -0.5, 0.0, 0.5]),
    'xi':    ('xi',    [0.1, 0.3, 0.6, 1.0]),
    'kappa': ('kappa', [0.5, 1.0, 3.0, 8.0]),
    'v0':    ('v0',    [0.01, 0.04, 0.09, 0.16]),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (label, (attr, values)) in zip(axes.flatten(), param_ranges.items()):
    for val in values:
        p = HestonParams(**{**vars(base), attr: val})
        if not p.is_valid(): continue
        m = HestonModel(p)
        vols = [m.implied_vol(S, K, T, r) for K in strikes]
        ax.plot(strikes/S, np.array(vols)*100, label=f'{attr}={val}')
    ax.set_title(f'Sensitivity to {attr}', fontsize=10)
    ax.set_xlabel('K/S'); ax.set_ylabel('IV (%)')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()